In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Nastavení úrovně optimalizace transpilátoru

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Skutečná kvantová zařízení jsou náchylná na šum a chyby Gate, takže optimalizace obvodů za účelem snížení jejich hloubky a počtu Gate může výrazně zlepšit výsledky získané spuštěním těchto obvodů.
Funkce [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) má jeden povinný poziční argument, `optimization_level`, který řídí, kolik úsilí Transpiler věnuje optimalizaci obvodů. Tento argument může být celé číslo nabývající jedné z hodnot 0, 1, 2 nebo 3.
Vyšší úrovně optimalizace generují více optimalizované obvody na úkor delší doby kompilace.
Následující tabulka vysvětluje optimalizace prováděné při každém nastavení.

<table>
  <thead>
    <tr>
      <th>Úroveň optimalizace</th>
      <th>Popis</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Bez optimalizace: typicky se používá pro charakterizaci hardwaru
        - Základní překlad
        - Layout/Routing: `TrivialLayout`, kde jsou jako fyzické vybrána stejná čísla Qubit jako virtuální a jsou vloženy SWAPy, aby to fungovalo (pomocí `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Lehká optimalizace:
        -   Layout/Routing: Layout se nejprve pokusí použít `TrivialLayout`. Jsou-li potřeba další SWAPy, je nalezen layout s minimálním počtem SWAPů pomocí `SabreSwap`, poté se použije `VF2LayoutPostLayout` k výběru nejlepších Qubit v grafu.
        -   `InverseCancellation`
        -   Optimalizace 1Q Gate
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Střední optimalizace:
          - Layout/Routing: Úroveň optimalizace 1 (bez triviální) + heuristická optimalizace s větší
        hloubkou prohledávání a počtem pokusů optimalizační funkce. Protože `TrivialLayout` se nepoužívá, není žádná snaha o použití stejných čísel fyzických a virtuálních Qubit.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Vysoká optimalizace:
        - Úroveň optimalizace 2 + heuristická optimalizace layoutu/routingu s větším úsilím a počtem pokusů
        - Resyntéza dvouqubitových bloků pomocí [Cartanovy KAK dekompozice](https://arxiv.org/abs/quant-ph/0507171).
        - Průchody porušující unitaritu:
          * `OptimizeSwapBeforeMeasure`: Přesouvá měření, aby se předešlo SWAPům
          * `RemoveDiagonalGatesBeforeMeasure`: Odstraňuje Gate před měřeními, která by měření neovlivnila
      </td>
    </tr>
  </tbody>
</table>

## Úroveň optimalizace v praxi
Protože dvouqubitové Gate jsou typicky nejvýznamnějším zdrojem chyb, můžeme přibližně kvantifikovat „hardwarovou efektivitu" transpilace počítáním počtu dvouqubitových Gate ve výsledném Circuit.
Zde vyzkoušíme různé úrovně optimalizace na vstupním Circuit sestávajícím z náhodné unitární matice následované Gate SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

V příkladech budeme používat mock Backend `FakeSherbrooke`. Nejprve transpilujme s úrovní optimalizace 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Transpilovaný Circuit má šest dvouqubitových Gate ECR.

Zopakujeme pro úroveň optimalizace 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Transpilovaný Circuit stále obsahuje šest Gate ECR, ale počet jednoqubitových Gate se snížil.

Zopakujeme pro úroveň optimalizace 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Tím se dosáhne stejných výsledků jako u úrovně optimalizace 1. Všimni si, že zvýšení úrovně optimalizace ne vždy přináší rozdíl.

Zopakujeme ještě jednou, s úrovní optimalizace 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Nyní jsou zde pouze tři Gate ECR. Tento výsledek získáme proto, že při úrovni optimalizace 3 se Qiskit pokouší znovu syntetizovat dvouqubitové bloky Gate a jakýkoli dvouqubitový Gate lze implementovat pomocí nejvýše tří Gate ECR. Ještě méně Gate ECR můžeme získat, nastavíme-li `approximation_degree` na hodnotu menší než 1, čímž umožníme Transpilátoru provádět aproximace, které mohou do dekompozice Gate vnést určitou chybu (viz [Běžně používané parametry pro transpilaci](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Tento Circuit má pouze dvě Gate ECR, ale jedná se o aproximovaný Circuit. Abychom pochopili, jak se jeho efekt liší od přesného Circuit, můžeme vypočítat věrnost (fidelitu) mezi unitárním operátorem, který tento Circuit implementuje, a přesnou unitární maticí. Před provedením výpočtu nejprve zredukujeme transpilovaný Circuit, který obsahuje 127 Qubit, na Circuit obsahující pouze aktivní Qubit, kterých je dva.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>